In [1]:
import pandas as pd
import sqlite3
import os

csv_path = 'g31_Equipment_Operators_Final_2.csv'
db_path = 'MB_&_FIC.db'

df = pd.read_csv(csv_path)
conn = sqlite3.connect(db_path)

equipment = df[['equipment_id', 'name', 'creation_date', 'equipment_type']].drop_duplicates()
equipment = equipment.rename(columns={'equipment_id': 'id', 'equipment_type': 'type'})
equipment.to_sql('Equipment', conn, if_exists='replace', index=False)

operators = df[['operator_id', 'title', 'category', 'birth_date']].drop_duplicates()
operators = operators.rename(columns={'operator_id': 'id'})
operators.to_sql('Operator', conn, if_exists='replace', index=False)

m_types = df[['maintenance_type_id', 'equipment_id']].dropna().drop_duplicates()
m_types = m_types.rename(columns={'maintenance_type_id': 'id'})
m_types['id'] = m_types['id'].astype(int)
m_types.to_sql('Maintenance_type', conn, if_exists='replace', index=False)

events = df[['maintenance_event_id', 'equipment_id', 'maintenance_type_id', 'maintenance_date', 'extra_info', 'maintenance_date_final']].dropna(subset=['maintenance_event_id'])
events = events.rename(columns={'maintenance_event_id': 'id'})
for col in ['id', 'equipment_id', 'maintenance_type_id']:
    events[col] = events[col].astype(int)
events.to_sql('Maintenance_event', conn, if_exists='replace', index=False)

usage = df[['equipment_id', 'operator_id', 'utilization_date', 'cost']]
usage.index = usage.index + 1
usage.to_sql('Equipment_operator', conn, if_exists='replace', index=True, index_label='id')

cursor = conn.cursor()
cursor.execute('''
CREATE TABLE IF NOT EXISTS User (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user TEXT NOT NULL UNIQUE,
    cargo TEXT,
    password TEXT NOT NULL
)
''')
conn.commit()

print("All tables created successfully:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

conn.close()


All tables created successfully:
[('Equipment',), ('Operator',), ('Maintenance_type',), ('Maintenance_event',), ('Equipment_operator',), ('User',), ('sqlite_sequence',)]
